# 📊 Notebook 2 - Transformação de Dados (Silver → Gold)

Este notebook realiza a transformação da camada **Silver** para a camada **Gold**, construindo uma base analítica orientada ao problema de recompra da Online Shop 2024.

### Problema analítico
O objetivo deste notebook é construir uma base gold que permita investigar o **comportamento de recompra dos clientes** da Online Shop 2024. As questões investigáveis do projeto envolvem recência, frequência e valor monetário de compras - dimensões centrais para entender retenção e segmentação de clientes em e-commerce.

### Unidade de análise da camada Gold
**Cada linha da base gold representa um cliente único**, identificado por `customer_id`. Todas as métricas são agregadas ao nível do cliente, sintetizando seu histórico completo de pedidos em um único registro. Essa escolha é coerente com perguntas do tipo "quais clientes têm maior risco de abandono?" ou "quais clientes geram mais receita?", que exigem uma visão consolidada por indivíduo.

### Metodologia RFM
A análise RFM (Recência, Frequência e Valor Monetário) é uma técnica consolidada em marketing analítico e CRM para segmentação de clientes com base em comportamento de compra passado. Suas três dimensões são:

- **Recência (R):** há quanto tempo o cliente realizou a última compra. Clientes que compraram recentemente têm maior probabilidade de responder positivamente a novas ofertas (Hughes, 1994).
- **Frequência (F):** quantas vezes o cliente comprou no período analisado. Alta frequência indica hábito de compra e vínculo com a plataforma.
- **Valor Monetário (M):** quanto o cliente gastou no total. Identifica os clientes de maior retorno financeiro para o negócio.

A classificação por quartis é a abordagem mais comum para criar scores RFM comparáveis entre si, permitindo combinar as três dimensões em um score único e segmentar a base em perfis de comportamento (Blattberg et al., 2008).

### Justificativa do recorte: apenas pagamentos "Completed"
A análise é restrita a pedidos com `transaction_status = 'Completed'` por uma razão conceitual: apenas pedidos com pagamento confirmado representam **receita efetivada** e **intenção de compra concluída**. Pedidos com status `'Pending'` ainda não geraram receita (o pagamento pode falhar) e pedidos `'Failed'` nunca geraram receita. Incluí-los distorceria métricas como frequência e valor monetário, pois contabilizariam intenções não concretizadas como comportamento de compra real. Essa é uma decisão padrão em análises de retenção e LTV (Lifetime Value) de clientes.

### Esquema da base Gold
A tabela abaixo descreve todas as colunas que compõem o dataset gold, organizadas por dimensão RFM:

| Coluna | Tipo | Dimensão | Descrição |
|---|---|---|---|
| `customer_id` | int | PK | Identificador único do cliente (chave primária) |
| `primeira_compra` | datetime | - | Data do primeiro pedido efetivado |
| `ultima_compra` | datetime | - | Data do pedido mais recente efetivado |
| `recencia_dias` | int | R | Dias desde a última compra até a data de referência |
| `score_recencia` | int (1–4) | R | Quartil de recência (4 = mais recente, 1 = mais antigo) |
| `frequencia` | int | F | Número de pedidos únicos efetivados |
| `score_frequencia` | int (1–4) | F | Quartil de frequência (4 = mais pedidos) |
| `is_recomprador` | bool | F | `True` se o cliente realizou 2 ou mais pedidos |
| `score_frequencia_monetario` | int (1–4) | F×M | Quartil de valor monetário (usado em análise cruzada com frequência) |
| `valor_monetario` | float | M | Soma total dos pedidos efetivados pelo cliente |
| `score_monetario` | int (1–4) | M | Quartil de valor monetário (4 = maior gasto) |
| `ticket_medio` | float | M | Valor médio por pedido (`valor_monetario / frequencia`) |
| `score_rfm` | int (3–12) | RFM | Soma dos três scores (R + F + M) |
| `segmento_cliente` | str | RFM | Segmento de comportamento derivado do `score_rfm` |

**Segmentos de cliente (baseados em `score_rfm`):**
| Score | Segmento | Interpretação |
|---|---|---|
| 10–12 | Muito Leal | Alto desempenho nas três dimensões |
| 7–9 | Leal | Bom engajamento geral |
| 4–6 | Em Risco | Frequência ou recência baixas |
| 3 | Inativo | Scores mínimos em todas as dimensões |

## 1. Imports e Configurações

In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from pathlib import Path

# === Caminhos ===
PATH_SILVER = Path('../datasets/dataset_silver/')
PATH_GOLD   = Path('../datasets/dataset_gold/')

PATH_GOLD.mkdir(parents=True, exist_ok=True)

print(f'Silver: {PATH_SILVER.resolve()}')
print(f'Gold:   {PATH_GOLD.resolve()}')

## 2. Carregamento do Dataset Silver

Lê os três arquivos Parquet da camada Silver necessários para a análise: `orders.parquet`, `customers.parquet` e `payment.parquet`.

In [ ]:
df_orders    = pd.read_parquet(PATH_SILVER / 'orders.parquet')
df_customers = pd.read_parquet(PATH_SILVER / 'customers.parquet')
df_payment   = pd.read_parquet(PATH_SILVER / 'payment.parquet')

print(f'orders    → {df_orders.shape}')
print(f'customers → {df_customers.shape}')
print(f'payment   → {df_payment.shape}')

df_orders.head()

## 3. Recorte Analítico: Pedidos com Pagamento Completado

Filtra os pedidos mantendo apenas aqueles cujo `transaction_status` é `'Completed'`. Pedidos com status `'Pending'` ou `'Failed'` não representam compras efetivadas. O join é feito entre `orders` e os `order_id` com pagamento completado, produzindo `df_orders_ok`, que passa a ser a base de cálculo de todas as métricas.

In [ ]:
print('Distribuição de transaction_status:')
print(df_payment['transaction_status'].value_counts())
print()

pagamentos_completos = df_payment[
    df_payment['transaction_status'] == 'Completed'
][['order_id']]

df_orders_ok = df_orders.merge(pagamentos_completos, on='order_id', how='inner')

print(f'Pedidos totais (Silver):            {len(df_orders):,}')
print(f'Pedidos com pagamento Completed:    {len(df_orders_ok):,}')
print(f'Pedidos excluídos (Pending/Failed): {len(df_orders) - len(df_orders_ok):,}')

df_orders_ok.head()

Agrega os pedidos efetivados por `customer_id`, produzindo uma linha por cliente com as métricas brutas que servirão para as transformações específicas de cada questão de pesquisa. As colunas geradas são `primeira_compra`, `ultima_compra`, `frequencia` e `valor_monetario`.

In [ ]:
df = df_orders_ok.groupby('customer_id').agg(
    primeira_compra = ('order_date',  'min'),
    ultima_compra   = ('order_date',  'max'),
    frequencia      = ('order_id',    'nunique'),
    valor_monetario = ('total_price', 'sum'),
).reset_index()

print(f'Clientes com ao menos 1 pedido efetivado: {len(df):,}')
print()
df.head()

## 4. Transformações para Q1

Para responder à Q1, são construídas duas métricas. A primeira é `recencia_dias`, que mede em dias corridos o tempo decorrido desde a última compra de cada cliente até a data de referência, definida como `max(order_date) + 1 dia`. O acréscimo de um dia garante que mesmo o cliente mais recente apresente ao menos 1 dia de inatividade, evitando recência zero e tornando o cálculo dos quartis consistente. Essa métrica testa H1, permitindo identificar quais clientes acumulam os maiores períodos sem compra e, portanto, compõem o maior risco de abandono definitivo da plataforma.

A segunda métrica é `score_recencia`, que classifica `recencia_dias` em 4 grupos iguais por quartil com pontuação ordinal de 1 a 4. A escala é invertida em relação à ordenação natural da variável: o quartil de menor inatividade recebe nota 4 e o de maior inatividade recebe nota 1, pois clientes mais recentes são mais valiosos para o negócio. Esse score permite, para testar H2, identificar o ponto de corte que separa os clientes ativos dos inativos e comparar esse limiar com o comportamento geral de compras ao longo do período.

In [ ]:
# Data de referência: último pedido do dataset + 1 dia
DATA_REFERENCIA = df_orders_ok['order_date'].max() + pd.Timedelta(days=1)
print(f'Data de referência: {DATA_REFERENCIA.date()}')
print()

# H1 - Recência absoluta em dias
df['recencia_dias'] = (DATA_REFERENCIA - df['ultima_compra']).dt.days

# H2 - Score ordinal por quartil (invertido: menor recência = nota mais alta)
df['score_recencia'] = pd.qcut(
    df['recencia_dias'],
    q=4,
    labels=[4, 3, 2, 1],
    duplicates='drop'
).astype(int)

print('Estatísticas de recencia_dias:')
print(df['recencia_dias'].describe())
print()
print('Distribuição de score_recencia (1 = mais inativo, 4 = mais recente):')
print(df['score_recencia'].value_counts().sort_index())
print()

## 5. Transformações para Q2

Para responder à Q2, a principal métrica é `frequencia`, já calculada na etapa de agregação como a contagem única de `order_id` por cliente. Essa variável indica quantos pedidos distintos cada cliente realizou ao longo do período analisado e é o insumo direto para testar H1: a hipótese de que o limite de corte para os clientes mais frequentes exigirá poucas repetições de compra, dado que a maioria da base realiza apenas um pedido. Sobre ela é aplicado `score_frequencia`, que segmenta os clientes em 4 grupos por quartil com notas de 1 a 4, sendo nota 4 reservada ao grupo de maior frequência. O parâmetro `duplicates='drop'` é necessário porque `frequencia` é discreta, muitos clientes podem ter o mesmo valor inteiro, impedindo a formação de 4 fronteiras distintas. Como variável derivada complementar, `is_recomprador` é criada como flag booleana (`True` para `frequencia >= 2`), oferecendo um recorte binário direto entre quem comprou uma única vez e quem demonstrou comportamento de retorno.

Para testar H2 de que clientes frequentes possuem CAC já amortizado e entregam maior retorno, é calculado `score_frequencia_monetario`, que classifica o `valor_monetario` de todos os clientes em 4 quartis. Esse score, cruzado com `score_frequencia`, permite verificar se os clientes de maior frequência também concentram os maiores gastos absolutos, evidenciando a diluição do custo de aquisição ao longo dos pedidos.

In [ ]:
# H1 - Score ordinal de frequência por quartil
frequencia_cut, bins = pd.qcut(
    df['frequencia'],
    q=4,
    retbins=True,
    duplicates='drop'
)
n_bins = len(bins) - 1
df['score_frequencia'] = pd.qcut(
    df['frequencia'],
    q=4,
    labels=list(range(1, n_bins + 1)),
    duplicates='drop'
).astype(int)

# H1 - Flag binária de recomprador (variável derivada complementar)
df['is_recomprador'] = df['frequencia'] >= 2

# H2 - Receita total por cliente classificada em quartis (todos os clientes)
df['score_frequencia_monetario'] = pd.qcut(
    df['valor_monetario'],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates='drop'
).astype(int)

print('Estatísticas de frequencia:')
print(df['frequencia'].describe())
print()
print('Distribuição de score_frequencia (1 = menos pedidos, 4 = mais pedidos):')
print(df['score_frequencia'].value_counts().sort_index())
print()
rc     = df['is_recomprador'].value_counts()
rc_pct = df['is_recomprador'].value_counts(normalize=True) * 100
print('Distribuição de is_recomprador:')
print(pd.DataFrame({'Clientes': rc, '%': rc_pct.round(1)}))

## 6. Transformações para Q3

Para responder à Q3, são construídas duas métricas centradas no retorno financeiro gerado por cada cliente. A primeira é `valor_monetario`, já calculada na etapa de agregação como a soma de `total_price` por cliente, e sobre ela é aplicado `score_monetario`, que segmenta todos os clientes em 4 quartis com notas de 1 a 4. Esse score testa H1, a hipótese de que uma parcela reduzida da base concentra a maior parte da receita, ao permitir comparar quanto do faturamento total está nas mãos do grupo de nota 4 em relação ao restante da base.

A segunda métrica é `ticket_medio`, calculada como `valor_monetario / frequencia`, ou seja, o valor médio que o cliente gasta a cada pedido. Ela testa H2, que questiona se clientes com maior gasto absoluto são necessariamente os que compram com mais frequência. Ao cruzar `ticket_medio` com `frequencia`, é possível identificar se o alto valor monetário de um cliente vem de muitas compras de valor baixo ou de poucas compras de alto valor, separando comportamentos de volume e de ticket.

In [ ]:
# H1 - Score ordinal monetário por quartil
df['score_monetario'] = pd.qcut(
    df['valor_monetario'],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates='drop'
).astype(int)

# H2 - Ticket médio por cliente
df['ticket_medio'] = df['valor_monetario'] / df['frequencia']

print('Estatísticas de valor_monetario:')
print(df['valor_monetario'].describe())
print()
print('Distribuição de score_monetario (1 = menor gasto, 4 = maior gasto):')
print(df['score_monetario'].value_counts().sort_index())
print()
print('Estatísticas de ticket_medio:')
print(df['ticket_medio'].describe())

## 7. Score RFM Combinado e Segmentação de Cliente

Combina os três scores individuais (R + F + M) em um `score_rfm` único, somando as notas de cada dimensão com peso igual (1:1:1). Em seguida, cria a coluna `segmento_cliente` classificando cada cliente em um dos quatro segmentos de negócio com base no score combinado:
- **Muito Leal** (score 10–12): alto desempenho nas três dimensões, clientes mais valiosos da base.
- **Leal** (score 7–9): bom engajamento geral, candidatos à fidelização.
- **Em Risco** (score 4–6): frequência ou recência baixas, necessitam de reativação.
- **Inativo** (score 3): scores mínimos em todas as dimensões, baixíssima probabilidade de retorno espontâneo.

In [ ]:
df['score_rfm'] = (
    df['score_recencia'] +
    df['score_frequencia'] +
    df['score_monetario']
)

def segmentar_cliente(score):
    if score >= 10:
        return 'Muito Leal'
    elif score >= 7:
        return 'Leal'
    elif score >= 4:
        return 'Em Risco'
    else:
        return 'Inativo'

df['segmento_cliente'] = df['score_rfm'].apply(segmentar_cliente)

print('Score RFM combinado (estatísticas):')
print(df['score_rfm'].describe())
print()
seg_dist = df['segmento_cliente'].value_counts()
seg_pct  = df['segmento_cliente'].value_counts(normalize=True) * 100
print('Distribuição de segmento_cliente:')
print(pd.DataFrame({'Clientes': seg_dist, '% Base': seg_pct.round(1)}))

## 8. Validação da Base Gold

Verifica a consistência da base construída antes da exportação: unicidade da chave primária `customer_id`, ausência de valores nulos nas colunas obrigatórias e validações lógicas das métricas calculadas (recência mínima de 1 dia, frequência mínima de 1 pedido, valores monetários positivos e scores dentro do intervalo esperado de 3 a 12).

In [ ]:
df_gold = df[[
    'customer_id',
    'primeira_compra', 'ultima_compra',
    'recencia_dias',   'score_recencia',
    'frequencia',      'score_frequencia',  'is_recomprador', 'score_frequencia_monetario',
    'valor_monetario', 'score_monetario',   'ticket_medio',
    'score_rfm', 'segmento_cliente',
]].copy()

print('=== VALIDAÇÃO FINAL - dataset Gold ===')
print(f'Shape: {df_gold.shape}')
print(f'customer_id únicos: {df_gold["customer_id"].nunique():,}')
print(f'Duplicatas na PK:   {df_gold["customer_id"].duplicated().sum()}')
print()

cols_obrigatorias = [
    'recencia_dias', 'score_recencia',
    'frequencia', 'score_frequencia', 'is_recomprador', 'score_frequencia_monetario',
    'valor_monetario', 'score_monetario', 'ticket_medio',
    'score_rfm', 'segmento_cliente',
]
nulos_obrig = df_gold[cols_obrigatorias].isnull().sum()
print('Nulos nas colunas obrigatórias:')
print(nulos_obrig[nulos_obrig > 0] if nulos_obrig.sum() > 0 else '✅ Nenhum valor nulo.')
print()

print('Verificações lógicas:')
print(f'  recencia_dias >= 1:      {(df_gold["recencia_dias"] >= 1).all()}')
print(f'  frequencia >= 1:         {(df_gold["frequencia"] >= 1).all()}')
print(f'  valor_monetario > 0:     {(df_gold["valor_monetario"] > 0).all()}')
print(f'  ticket_medio > 0:        {(df_gold["ticket_medio"] > 0).all()}')
print(f'  score_rfm entre 3 e 12:  {df_gold["score_rfm"].between(3, 12).all()}')
print()

df_gold.dtypes

## 9. Geração do Dataset Gold

Salva o `DataFrame` `df_gold` em um arquivo Parquet chamado `dataset_gold.parquet` na pasta `dataset_gold/`.

In [ ]:
df_gold.to_parquet(PATH_GOLD / 'dataset_gold.parquet', index=False)

print('✅ dataset_gold.parquet salvo com sucesso!')
print(f'   Registros: {len(df_gold):,}')
print(f'   Colunas:   {len(df_gold.columns)}')
print()
print(f'   Destino: {PATH_GOLD.resolve()}')

## 10. Comparação Silver × Gold

| Aspecto | Silver | Gold |
|---|---|---|
| Número de tabelas | 8 tabelas separadas | 1 tabela unificada |
| Granularidade | Registros transacionais (pedidos, itens, pagamentos…) | 1 linha por cliente |
| Total de registros | 15.000 pedidos, 10.000 clientes, 20.000 itens… | N clientes com ao menos 1 pedido efetivado |
| Colunas | Colunas originais limpas | Métricas derivadas (RFM scores, segmento, ticket médio…) |
| Propósito | Base limpa e confiável, preserva toda a informação | Base analítica pronta para responder as questões do projeto |